In [7]:
import numpy as np
import matplotlib.pyplot as plt

In [8]:
import labmate
from labmate.acquisition_notebook import AcquisitionAnalysisManager

from datetime import datetime
from pathlib import Path

In [9]:
import pyvisa as pv

In [10]:
from utils.naming import make_run_dir
from utils.pm100a import record_pm100a

In [11]:
import os
from sys import path
import struct
import time

#### Add DLL to local path

In [4]:
path.append(r"D:/NKT Photonics/Examples/DLL_Example_Python")

In [ ]:
from NKTP_DLL import *
from utils.nkt import read_power_and_temp

Loading x64 DLL from: D:\NKT Photonics\\NKTPDLL\x64\NKTPDLL.dll


In [6]:
DATA_DIR = "data/nkt"
os.makedirs(DATA_DIR, exist_ok=True)

# Power fluctuations

We record the power fluctuations at the laser output (from NKT data) with the power at a given point of the experiment table

In [ ]:
# Check available VISA resources
rm = pv.ResourceManager()
print(rm.list_resources())

In [ ]:
# Connect to power meter via PyVISA
# you can use the tool Power Meter Driver Switcher to switch between the two drivers, the PM100D.dll driver and the new TLPM.dll driver. 
# Pyvisa may not recognize the PM100A with the WinUSB driver, so you may need to switch to the Visa driver. 
# PM100A
# pm = rm.open_resource('USB0::0x1313::0x8079::P1007388::INSTR')  # PANDA powermeter
# print(pm.query('*IDN?'))

# PM100D Musiqs
# pm100d = rm.open_resource('USB0::0x1313::0x8078::P0008159::INSTR')  # PESTO powermeter

pm = rm.open_resource('USB0::0x1313::0x8078::P0025589::INSTR')  # Musiqs powermeter

print(pm.query('*IDN?'))

In [ ]:
MEAS = "laser_fluctuation"
SAMPLE = "shg_injection"

RUN_DIR = make_run_dir(data_dir=DATA_DIR, meas=MEAS, sample=SAMPLE)
ACQ_CELL = f"{MEAS}_{SAMPLE}"

In [ ]:
ACQ_DURATION = 60 * 60      # 1 hour
PM_SAMPLE_DELAY = 0.1       # 100 ms

timestamps = []
pm_power_mW = []
laser_power = []
laser_temp = []

t0 = time.perf_counter()

# Acquisition and analysis
aqm = AcquisitionAnalysisManager(RUN_DIR)
aqm.acquisition_cell(ACQ_CELL)

while True:
    t = time.perf_counter() - t0

    if t >= ACQ_DURATION:
        break

    # Read both instruments as close together as possible
    pm_value =  float(pm.query("MEAS:POW?").strip())
    laser_pwr, laser_temp_val = read_power_and_temp()

    timestamps.append(t)
    pm_power_mW.append(pm_value * 1e3) # convert to mW
    laser_power.append(laser_pwr)
    laser_temp.append(laser_temp_val)

    time.sleep(PM_SAMPLE_DELAY)

timestamps = np.array(timestamps)
pm_power_mW = np.array(pm_power_mW)
laser_power = np.array(laser_power)
laser_temp = np.array(laser_temp)

aqm.save_acquisition(timestamps=timestamps, pm_power_mW=pm_power_mW, laser_power=laser_power, laser_temp=laser_temp)